In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer

from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix
)

import warnings
warnings.filterwarnings("ignore")

print("Libraries imported successfully!")
import zipfile
import os

# Name of ZIP file
zip_file = "dim_events.zip"

# Extract the ZIP
with zipfile.ZipFile(zip_file, 'r') as zip_ref:
    zip_ref.extractall("quickcart_data")

print("Files extracted successfully!")
#5DATASETS
stores = pd.read_csv("QuickCart_Data/dim_stores.csv")
skus = pd.read_csv("QuickCart_Data/dim_skus.csv")
suppliers = pd.read_csv("QuickCart_Data/dim_suppliers.csv")
events = pd.read_csv("QuickCart_Data/dim_events.csv")
inventory = pd.read_csv("QuickCart_Data/fact_inventory_daily.csv")
print("Stores:", stores.shape)
print("SKUs:", skus.shape)
print("Suppliers:", suppliers.shape)
print("Events:", events.shape)
print("Inventory:", inventory.shape)
print("STORES:")
print(stores.columns.tolist())

print("\nSKUS:")
print(skus.columns.tolist())

print("\nSUPPLIERS:")
print(suppliers.columns.tolist())

print("\nEVENTS:")
print(events.columns.tolist())

print("\nINVENTORY:")
print(inventory.columns.tolist())
inventory["date"] = pd.to_datetime(inventory["date"])
events["date"] = pd.to_datetime(events["date"])

print("Inventory dates:")
print(inventory["date"].min(), "to", inventory["date"].max())

print("\nEvent dates:")
print(events["date"].min(), "to", events["date"].max())
print("STORES")
display(stores.isnull().sum())

print("SKUS")
display(skus.isnull().sum())

print("SUPPLIERS")
display(suppliers.isnull().sum())

print("EVENTS")
display(events.isnull().sum())

print("INVENTORY")
display(inventory.isnull().sum())
print("Stores duplicates:", stores.duplicated().sum())
print("SKUs duplicates:", skus.duplicated().sum())
print("Suppliers duplicates:", suppliers.duplicated().sum())
print("Events duplicates:", events.duplicated().sum())
print("Inventory duplicates:", inventory.duplicated().sum())
print("Missing values:")

display(stores.isnull().sum())
display(skus.isnull().sum())
display(suppliers.isnull().sum())
display(events.isnull().sum())
display(inventory.isnull().sum())

print("\nDuplicates:")

print("Stores:", stores.duplicated().sum())
print("SKUs:", skus.duplicated().sum())
print("Suppliers:", suppliers.duplicated().sum())
print("Events:", events.duplicated().sum())
print("Inventory:", inventory.duplicated().sum())
#CLEAN & MERGE
stores["city_display"] = (
    stores["city_display"]
    .astype(str)
    .str.strip()
    .str.title()
)
suppliers["reliability_score"] = pd.to_numeric(
    suppliers["reliability_score"],
    errors="coerce"
)

print("Missing supplier reliability:",
      suppliers["reliability_score"].isnull().sum())
df = inventory.merge(
    stores,
    on="store_id",
    how="left"
)

df = df.merge(
    skus,
    on="sku_id",
    how="left",
    suffixes=("", "_sku")
)

df = df.merge(
    suppliers,
    on="supplier_id",
    how="left",
    suffixes=("", "_supplier")
)

df = df.merge(
    events,
    on="date",
    how="left",
    suffixes=("", "_event")
)

print("Final merged dataset shape:", df.shape)
display(df.head())
print(df.columns.tolist())
print(df["stockout_risk"].value_counts())
print(
    df["stockout_risk"]
    .value_counts(normalize=True)
    .mul(100)
    .round(2)
)
#EDA
#STOCKOUT RISK DISTRIBUTION
plt.figure(figsize=(7,5))

sns.countplot(
    data=df,
    x="stockout_risk"
)

plt.title("Stockout Risk Distribution")
plt.xlabel("Stockout Risk")
plt.ylabel("Number of Records")
plt.show()
#FESTIVAL VS NON-FESTIVAL
print(
    df.groupby("event_type")["stockout_risk"]
    .value_counts(normalize=True)
    .mul(100)
    .round(2)
)
plt.figure(figsize=(10,5))

sns.countplot(
    data=df,
    x="category",
    hue="stockout_risk"
)

plt.title("Stockout Risk by Product Category")
plt.xticks(rotation=45)
plt.show()
#RISK BY STORE
plt.figure(figsize=(10,5))
sns.countplot(
    data=df,
    x="store_id",
    hue="stockout_risk"
)

plt.title("Stockout Risk by Store")
plt.show()
#PERISHABLE VS NON-PERISHABLE
print(
    df.groupby("is_perishable")["stockout_risk"]
    .value_counts(normalize=True)
    .mul(100)
    .round(2)
)
#FEATURE ENGINEERING
#REorder gap
df["reorder_gap"] = (
    df["reorder_point"] - df["closing_stock"]
)
#DAYS OF COVER RATIO
df["days_of_cover_ratio"] = (
    df["days_of_cover"] /
    df["lead_time_days_expected"].replace(0, np.nan)
)
#SUPPILER REALIABILITY
df["supplier_reliability_clean"] = (
    df["reliability_score"]
)

df["supplier_reliability_clean"] = (
    df["supplier_reliability_clean"]
    .fillna(
        df.groupby("category")["supplier_reliability_clean"]
        .transform("median")
    )
)

df["supplier_reliability_clean"] = (
    df["supplier_reliability_clean"]
    .fillna(df["supplier_reliability_clean"].median())
)
#DATA FEATURES
df["day_of_month"] = df["date"].dt.day
df["day_of_week"] = df["date"].dt.dayofweek
df["is_weekend"] = (
    df["day_of_week"] >= 5
).astype(int)
df["is_perishable_flag"] = (
    df["is_perishable"]
    .astype(str)
    .str.lower()
    .isin(["yes", "true", "1"])
    .astype(int)
)
Festival flag
df["festive_relevant_flag"] = (
    df["festive_relevant"]
    .astype(str)
    .str.lower()
    .isin(["yes", "true", "1"])
    .astype(int)
)

print("Feature engineering completed!")

print("\nNew features:")
print([
    "reorder_gap",
    "days_of_cover_ratio",
    "supplier_reliability_clean",
    "day_of_month",
    "day_of_week",
    "is_weekend",
    "is_perishable_flag",
    "festive_relevant_flag"
])
#ML
train_df = df[df["date"] < "2026-10-24"].copy()

test_df = df[df["date"] >= "2026-10-24"].copy()

print("Training data:", train_df.shape)
print("Testing data:", test_df.shape)
#ML FEATURES
features = [
    "opening_stock",
    "units_demanded",
    "units_sold",
    "closing_stock",
    "reorder_point",
    "lead_time_days_expected",
    "sales_velocity_7d",
    "days_of_cover",
    "reorder_gap",
    "days_of_cover_ratio",
    "supplier_reliability_clean",
    "day_of_month",
    "day_of_week",
    "is_weekend",
    "is_perishable_flag",
    "category",
    "popularity_tier",
    "store_size",
    "city",
    "event_type"
]

target = "stockout_risk"

X_train = train_df[features]
y_train = train_df[target]

X_test = test_df[features]
y_test = test_df[target]

print("Training features:", X_train.shape)
print("Testing features:", X_test.shape)
#PREPROCESSING
numeric_features = [
    "opening_stock",
    "units_demanded",
    "units_sold",
    "closing_stock",
    "reorder_point",
    "lead_time_days_expected",
    "sales_velocity_7d",
    "days_of_cover",
    "reorder_gap",
    "days_of_cover_ratio",
    "supplier_reliability_clean",
    "day_of_month",
    "day_of_week",
    "is_weekend",
    "is_perishable_flag"
]

categorical_features = [
    "category",
    "popularity_tier",
    "store_size",
    "city",
    "event_type"
]
numeric_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ]
)

categorical_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore"))
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features)
    ]
)
#BASELINE
baseline = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("model", DummyClassifier(strategy="most_frequent"))
    ]
)

baseline.fit(X_train, y_train)

baseline_pred = baseline.predict(X_test)

print(
    "Baseline Accuracy:",
    accuracy_score(y_test, baseline_pred)
)
#LOGESTIC REGRESSION
logistic_model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        (
            "model",
            LogisticRegression(
                max_iter=2000,
                class_weight="balanced"
            )
        )
    ]
)

logistic_model.fit(X_train, y_train)

logistic_pred = logistic_model.predict(X_test)

print(
    classification_report(
        y_test,
        logistic_pred
    )
)
#RANDOMFOREST
rf_model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        (
            "model",
            RandomForestClassifier(
                n_estimators=300,
                random_state=42,
                class_weight="balanced",
                n_jobs=-1,
                min_samples_leaf=2
            )
        )
    ]
)

rf_model.fit(X_train, y_train)

rf_pred = rf_model.predict(X_test)

print(
    classification_report(
        y_test,
        rf_pred
    )
)
#EVALUATE MODELS
results = pd.DataFrame({
    "Model": [
        "Baseline",
        "Logistic Regression",
        "Random Forest"
    ],
    
    "Accuracy": [
        accuracy_score(y_test, baseline_pred),
        accuracy_score(y_test, logistic_pred),
        accuracy_score(y_test, rf_pred)
    ],
    
    "Precision": [
        precision_score(
            y_test,
            baseline_pred,
            average="weighted",
            zero_division=0
        ),
        precision_score(
            y_test,
            logistic_pred,
            average="weighted",
            zero_division=0
        ),
        precision_score(
            y_test,
            rf_pred,
            average="weighted",
            zero_division=0
        )
    ],
    
    "Recall": [
        recall_score(
            y_test,
            baseline_pred,
            average="weighted",
            zero_division=0
        ),
        recall_score(
            y_test,
            logistic_pred,
            average="weighted",
            zero_division=0
        ),
        recall_score(
            y_test,
            rf_pred,
            average="weighted",
            zero_division=0
        )
    ],
    
    "F1 Score": [
        f1_score(
            y_test,
            baseline_pred,
            average="weighted",
            zero_division=0
        ),
        f1_score(
            y_test,
            logistic_pred,
            average="weighted",
            zero_division=0
        ),
        f1_score(
            y_test,
            rf_pred,
            average="weighted",
            zero_division=0
        )
    ]
})

display(results)
#CONFUSION MATRIX
cm = confusion_matrix(y_test, rf_pred)

plt.figure(figsize=(7,5))

sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues"
)

plt.title("Random Forest Confusion Matrix")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.show()
#IMPORTANCE'
rf_preprocessor = rf_model.named_steps["preprocessor"]
rf_classifier = rf_model.named_steps["model"]

feature_names = rf_preprocessor.get_feature_names_out()

importance = rf_classifier.feature_importances_

feature_importance = pd.DataFrame({
    "Feature": feature_names,
    "Importance": importance
})

feature_importance = (
    feature_importance
    .sort_values("Importance", ascending=False)
)

display(feature_importance.head(20))
#PLOT FEATURE
top_features = feature_importance.head(15)

plt.figure(figsize=(10,6))

sns.barplot(
    data=top_features,
    x="Importance",
    y="Feature"
)

plt.title("Top 15 Features Influencing Stockout Risk")
plt.show()
#PREDCTIONS
predictions = test_df[
    [
        "date",
        "store_id",
        "sku_id",
        "stockout_risk"
    ]
].copy()

predictions["predicted_stockout_risk"] = rf_pred

display(predictions.head(20))
#SAVE
df.to_csv(
    "QuickCart_Final_Merged_Data.csv",
    index=False
)

results.to_csv(
    "QuickCart_Model_Comparison.csv",
    index=False
)

predictions.to_csv(
    "QuickCart_Predictions.csv",
    index=False
)

feature_importance.to_csv(
    "QuickCart_Feature_Importance.csv",
    index=False
)

print("All result files saved successfully!")


SyntaxError: invalid syntax (233942768.py, line 246)